In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "validation").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from validation.notebook_bootstrap import bootstrap

bedrock_model_arn, load_workshop_state, persist_workshop_state_file = bootstrap()


# Metadata filtering usando Amazon Bedrock Knowledge Bases
Este notebook fornece um exemplo de código passo a passo para o recurso de 'metadata filtering' do Amazon Bedrock Knowledge Bases.

Usando o recurso de metadata filtering, você pode melhorar os resultados de busca pré-filtrando suas recuperações dos vector stores.
Para mais detalhes sobre esse recurso, leia este [blog](https://aws.amazon.com/blogs/machine-learning/amazon-bedrock-knowledge-bases-now-supports-metadata-filtering-to-improve-retrieval-accuracy/).

## 1. Importar as bibliotecas necessárias
O primeiro passo é instalar os pacotes de pré-requisitos.

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../requirements.txt --no-deps --quiet
%pip install -r ../requirements.txt --upgrade --quiet

In [ ]:
# Kernel restart is intentionally skipped in corrected notebooks.


In [ ]:
import botocore
botocore.__version__

In [ ]:
import os
import sys
import time
import boto3
import logging
import pprint
import json

# Set the path to import module
from pathlib import Path
current_path = Path().resolve()
current_path = current_path.parent
if str(current_path) not in sys.path:
    sys.path.append(str(current_path))
# Print sys.path to verify
# print(sys.path)

from utils.knowledge_base import BedrockKnowledgeBase

In [ ]:
#Clients
s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region =  session.region_name
account_id = sts_client.get_caller_identity()["Account"]
bedrock_agent_client = boto3.client('bedrock-agent')
bedrock_agent_runtime_client = boto3.client('bedrock-agent-runtime') 
logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)
region, account_id

In [ ]:
import uuid

suffix = uuid.uuid4().hex[:12]
knowledge_base_name = 'metadata-filtering-kb'
knowledge_base_description = "Knowledge Base metadata filtering."
bucket_name = f'{knowledge_base_name}-{suffix}'
foundation_model = os.getenv("BEDROCK_TEXT_MODEL_ID", "us.anthropic.claude-haiku-4-5-20251001-v1:0")

# Define data sources
data_source=[{"type": "S3", "bucket_name": bucket_name}]


## 2 - Criar knowledge bases com estratégia de fixed chunking
Vamos começar criando uma [Amazon Bedrock Knowledge Bases](https://aws.amazon.com/bedrock/knowledge-bases/) para armazenar dados de video games no formato csv. Knowledge Bases permitem integração com diferentes bancos de dados vetoriais incluindo [Amazon OpenSearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/), [Amazon Aurora](https://aws.amazon.com/rds/aurora/), [Pinecone](http://app.pinecone.io/bedrock-integration), [Redis Enterprise]() e [MongoDB Atlas](). Para este exemplo, vamos integrar a knowledge base com o Amazon OpenSearch Serverless. Para isso, usaremos a classe helper `BedrockKnowledgeBase` que criará a knowledge base e todos os seus pré-requisitos:
1. IAM roles e policies
2. S3 bucket
3. Políticas de encryption, network e data access do Amazon OpenSearch Serverless
4. Coleção Amazon OpenSearch Serverless
5. Índice vetorial do Amazon OpenSearch Serverless
6. Knowledge base
7. Data source da Knowledge base

Vamos criar uma knowledge base usando a estratégia de fixed chunking.

Você pode escolher diferentes estratégias de chunking alterando os valores do parâmetro abaixo:
```
"chunkingStrategy": "FIXED_SIZE | NONE | HIERARCHICAL | SEMANTIC"
```

In [ ]:
knowledge_base_metadata = BedrockKnowledgeBase(
    kb_name=f'{knowledge_base_name}-{suffix}',
    kb_description=knowledge_base_description,
    data_sources=data_source, 
    chunking_strategy = "FIXED_SIZE", 
    suffix = suffix
)

# Keep resources from this notebook discoverable by the workshop cleanup.
s3_client.put_bucket_tagging(
    Bucket=bucket_name,
    Tagging={"TagSet": [{"Key": "workshop-kb", "Value": "true"}]},
)


### 2.1 Baixar o dataset de video games e fazer upload para o Amazon S3

Agora que criamos a knowledge base, vamos populá-la com o dataset `video_games` na KB. Esses dados estão sendo baixados [daqui](https://aws-blogs-artifacts-public.s3.amazonaws.com/ML-16482/30_generated_video_game_records.zip). Os dados são sobre video games fictícios contendo informações como título, descrição, gênero, ano, editora e pontuação para cada video game.

In [ ]:
import zipfile
from pathlib import Path
from time import sleep
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen


def download_file(url, filename, attempts=3, timeout=60):
    destination = Path(filename)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name(destination.name + ".part")
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(request, timeout=timeout) as response:
                status = getattr(response, "status", 200)
                if status >= 400:
                    raise RuntimeError(f"HTTP {status} while downloading {url}")
                with temporary.open("wb") as output:
                    while True:
                        chunk = response.read(1024 * 1024)
                        if not chunk:
                            break
                        output.write(chunk)
            if temporary.stat().st_size == 0:
                raise RuntimeError("Downloaded file is empty")
            temporary.replace(destination)
            print(f"File downloaded successfully: {destination}")
            return destination
        except (HTTPError, URLError, TimeoutError, OSError, RuntimeError) as error:
            last_error = error
            temporary.unlink(missing_ok=True)
            if attempt < attempts:
                sleep(2 ** (attempt - 1))

    raise RuntimeError(f"Could not download {url} after {attempts} attempts") from last_error


zip_path = download_file(
    "https://aws-blogs-artifacts-public.s3.amazonaws.com/ML-16482/30_generated_video_game_records.zip",
    "./30_generated_video_game_records.zip",
)

if not zipfile.is_zipfile(zip_path):
    raise ValueError(f"Downloaded file is not a valid ZIP archive: {zip_path}")

# Unzip the file content into the video_game folder.
with zipfile.ZipFile(zip_path, "r") as zipf:
    if zipf.testzip() is not None:
        raise ValueError("Downloaded ZIP archive is corrupted")
    csv_files = [
        item for item in zipf.infolist()
        if not item.filename.startswith("__MACOSX/") and item.filename.endswith(".csv")
    ]
    if not csv_files:
        raise FileNotFoundError("The ZIP archive does not contain a CSV dataset")
    for csv_file in csv_files:
        zipf.extract(csv_file, "./")


Vamos fazer o upload dos dados de video games disponíveis na pasta `video_game` para o s3.

In [ ]:
def upload_directory(path, bucket_name):
        for root,dirs,files in os.walk(path):
            for file in files:
                if not file.startswith('.DS_Store'):
                    file_to_upload = os.path.join(root,file)
                    print(f"uploading file {file_to_upload} to {bucket_name}")
                    s3_client.upload_file(file_to_upload,bucket_name,file)

upload_directory("video_game", bucket_name)

Agora iniciamos o ingestion job.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_metadata.start_ingestion_job()

Por fim, salvamos o Knowledge Base Id para testar a solução em uma etapa posterior.

In [ ]:
kb_id_metadata = knowledge_base_metadata.get_knowledge_base_id()

### 2.2 Consultar a Knowledge Base com a API Retrieve and Generate - sem metadata

Vamos testar a knowledge base usando a API [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html). Com essa API, o Bedrock cuida de recuperar as referências necessárias da knowledge base e gerar a resposta final usando um foundation model do Bedrock.

'''
query = "A strategy game with cool graphic with score of 9.0"
'''

Resultados esperados:
    * Fantasy Kingdoms: Chronicles of Eldoria é um jogo de strategy RPG com pontuação de 9.0.

In [ ]:
query = "A strategy game with cool graphic with score of 9.0"

In [ ]:
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_metadata,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5
                } 
            }
        }
    }
)

pprint.pp(response['output']['text'])

#### 2.3 Preparar metadata para ingestão

In [ ]:
import csv
import json
import pandas as pd

def generate_matadata(data_dir , metadata_fields):
    # Define the metadata attributes
    metadata_attributes = metadata_fields

    # Loop through all CSV files in the directory
    for filename in os.listdir(data_dir):
        filename= f'{data_dir}/{filename}'
        if filename.endswith(".csv"):
            # Read the CSV file
            df = pd.read_csv(filename)
            df["Id"] = [os.path.basename(filename)]
            
            # Extract the metadata attributes
            metadata = {k:v[0] for k,v in df[metadata_attributes].to_dict(orient='list').items()}
            # reorder the keys
            metadata = {key: metadata[key] for key in metadata_attributes}
            
            # Create a JSON object
            json_data = {"metadataAttributes": metadata}
            
            
            # Write the JSON object to a file
            with open(f"{filename.replace('.csv', '.csv.metadata.json')}", "w") as f:
                json.dump(json_data, f)

In [ ]:
data_dir = './video_game'
metadata_fields = ["Id", "genres", "year", "publisher", "score"]

generate_matadata(data_dir, metadata_fields)

In [ ]:
# upload metadata file to S3
upload_directory("video_game", bucket_name)

In [ ]:
# delete metadata files from local
data_dir = './video_game'
for filename in os.listdir(data_dir):
    filename= f'{data_dir}/{filename}'
    if filename.endswith(".csv.metadata.json"):
        os.remove(filename)

Agora inicie o ingestion job. Como estamos usando os mesmos documentos usados no fixed chunking, estamos pulando a etapa de upload dos documentos para o bucket s3.

In [ ]:
# ensure that the kb is available
time.sleep(30)
# sync knowledge base
knowledge_base_metadata.start_ingestion_job()

### 2.4 Consultar a Knowledge Base com a API Retrieve and Generate - com metadata

Criar o filter

In [ ]:
one_group_filter= {
    "andAll": [
        {
            "equals": {
                "key": "genres",
                "value": "Strategy"
            }
        },
        {
            "greaterThanOrEquals": {
                "key": "score",
                "value": 9.0
            }
        }
    ]
}

Passe o filter para `retrievalConfiguration` da API [**retrieve_and_generate**](https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/bedrock-agent-runtime/client/retrieve_and_generate.html).

In [ ]:
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        "text": query
    },
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            'knowledgeBaseId': kb_id_metadata,
            "modelArn": bedrock_model_arn(foundation_model, region),
            "retrievalConfiguration": {
                "vectorSearchConfiguration": {
                    "numberOfResults":5,
                    "filter": one_group_filter
                } 
            }
        }
    }
)

print(response['output']['text'])

Como você pode ver, com a API retrieve and generate obtemos a resposta final diretamente. Agora vamos observar as citations da API `RetrieveAndGenerate`. Além disso, vamos observar os chunks recuperados e as citations retornadas pelo modelo ao gerar a resposta. Quando fornecemos o contexto relevante ao foundation model junto com a query, é muito provável que ele gere uma resposta de alta qualidade.

In [ ]:
# response_metadata = response['citations'][0]['retrievedReferences']
# print("# of citations or chunks used to generate the response: ", len(response_metadata))
# def citations_rag_print(response_ret):
# #structure 'retrievalResults': list of contents. Each list has content, location, score, metadata
#     for num,chunk in enumerate(response_ret,1):
#         print(f'Chunk {num}: ',chunk['content']['text'],end='\n'*2)
#         print(f'Chunk {num} Location: ',chunk['location'],end='\n'*2)
#         print(f'Chunk {num} Metadata: ',chunk['metadata'],end='\n'*2)

# citations_rag_print(response_metadata)

### Limpeza (Clean up)
Certifique-se de descomentar e executar as células abaixo para excluir os recursos criados neste notebook. Se você planeja executar o notebook `dynamic-metadata-filtering` na seção `03-advanced-concepts`, volte aqui depois para excluir os recursos.

In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")


In [ ]:
# Cleanup is intentionally deferred to full_cleanup.ipynb.
print("Cleanup deferred to full_cleanup.ipynb.")
